In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import platform
from scipy.stats import chi2_contingency

if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

user = pd.read_csv('../../data/processed/01_user_profile_preprocessed_v2.csv', encoding='utf-8-sig')
event = pd.read_csv('../../data/processed/02_event_log_preprocessed_v2.csv', encoding='utf-8-sig')

In [34]:
user['signup_date'] = pd.to_datetime(user['signup_date'])
event['event_time'] = pd.to_datetime(event['event_time'])
event['event_date'] = pd.to_datetime(event['event_date'])

In [35]:
# 로그 장애 기간 제외
event_clean = event[event['is_log_issue_period'] == False]

In [36]:
# D30 계산용 세팅
event_with_signup = event_clean.merge(
    user[['user_id', 'signup_date']],
    on='user_id', how='left'
)
event_with_signup['days_since_signup'] = (
    event_with_signup['event_date'] - 
    event_with_signup['signup_date']
).dt.days

app_launch = event_with_signup[event_with_signup['event_type'] == '앱실행']

print("로드 완료!")
print(f"전체 이벤트: {len(event_clean):,}건")

로드 완료!
전체 이벤트: 1,736,400건


통계량이 크다 
→ 관측값과 기댓값 차이가 크다
→ 귀무가설이 맞을 가능성이 낮다
→ p-value가 작아진다

통계량이 작다
→ 관측값과 기댓값 차이가 작다
→ 귀무가설이 맞을 가능성이 높다
→ p-value가 커진다

### 1. 가설설정
- 귀무가설(H0): 기능 경험 여부에 따라 D30 리텐션에 차이가 없다
- 대립가설(H1): 기능 경험 여부에 따라 D30 리텐션에 차이가 있다

- 유의수준: α = 0.05
- → p-value < 0.05 이면 귀무가설 기각 (차이 있음)

기능 경험 여부 → 범주형 (경험/미경험)
D30 리텐션 여부 → 범주형 (복귀/미복귀)

In [37]:
# D30 계산 함수
def get_d30_retention(user_ids, app_launch_df, user_df):
    total = len(user_ids)
    d30_users = app_launch_df[
        (app_launch_df['user_id'].isin(user_ids)) &
        (app_launch_df['days_since_signup'] == 30)
    ]['user_id'].nunique()
    return total, d30_users, round(d30_users / total * 100, 1)

# 기능별 경험 유저 추출
activity_types = {
    '수면기록': '수면기록',
    '운동기록': '운동기록',
    '식단기록': '식단기록',
    '마음챙김': '마음챙김',
    '챌린지참여': '챌린지참여'
}

all_users = set(user['user_id'])
result = []

for name, event_type in activity_types.items():
    exp_users = set(event_clean[event_clean['event_type'] == event_type]['user_id'])
    non_users = all_users - exp_users

    t1, d1, r1 = get_d30_retention(exp_users, app_launch, user)
    t2, d2, r2 = get_d30_retention(non_users, app_launch, user)

    # 카이제곱 검정
    table = [[d1, t1-d1], [d2, t2-d2]]
    chi2, p, _, _ = chi2_contingency(table)

    result.append({
        '기능': name,
        '경험유저수': t1, '경험D30': r1,
        '미경험유저수': t2, '미경험D30': r2,
        'p-value': round(p, 4),
        '유의미': 'O' if p < 0.05 else 'X'
    })

print(pd.DataFrame(result).to_string())

      기능  경험유저수  경험D30  미경험유저수  미경험D30  p-value 유의미
0   수면기록  11432   25.7    1068     0.0      0.0   O
1   운동기록  10374   28.4    2126     0.0      0.0   O
2   식단기록  10217   28.8    2283     0.0      0.0   O
3   마음챙김  10795   27.3    1705     0.0      0.0   O
4  챌린지참여   9321   31.4    3179     0.4      0.0   O


In [38]:
for name, event_type in activity_types.items():
    exp_users = set(event_clean[event_clean['event_type'] == event_type]['user_id'])
    non_users = all_users - exp_users

    t1, d1, r1 = get_d30_retention(exp_users, app_launch, user)
    t2, d2, r2 = get_d30_retention(non_users, app_launch, user)

    table = [[d1, t1-d1], [d2, t2-d2]]
    chi2, p, dof, expected = chi2_contingency(table)

    df = pd.DataFrame({
        '구분': [f'{name} 경험', f'{name} 미경험'],
        '전체유저수': [t1, t2],
        'D30 유저수': [d1, d2],
        'D30 리텐션(%)': [r1, r2],
    })

    # 검정 결과는 따로 출력
    stat_df = pd.DataFrame({
        '카이제곱 통계량': [round(chi2, 4)],
        '자유도': [dof],
        'p-value': [f'{p:.2e}'],
        '결론': ['유의미 O' if p < 0.05 else '유의미 X']
    })

    print(f"\n=== {name} ===")
    display(df)
    display(stat_df)
    print("-" * 50)


=== 수면기록 ===


,구분,전체유저수,D30 유저수,D30 리텐션(%)
0,수면기록 경험,11432,2943,25.7
1,수면기록 미경험,1068,0,0.0


,카이제곱 통계량,자유도,p-value,결론
0,358.1779,1,7.02e-80,유의미 O


--------------------------------------------------

=== 운동기록 ===


,구분,전체유저수,D30 유저수,D30 리텐션(%)
0,운동기록 경험,10374,2943,28.4
1,운동기록 미경험,2126,0,0.0


...

,카이제곱 통계량,자유도,p-value,결론
0,606.454,1,6.61e-134,유의미 O


--------------------------------------------------

=== 챌린지참여 ===


,구분,전체유저수,D30 유저수,D30 리텐션(%)
0,챌린지참여 경험,9321,2930,31.4
1,챌린지참여 미경험,3179,13,0.4


,카이제곱 통계량,자유도,p-value,결론
0,1265.893,1,2.92e-277,유의미 O


--------------------------------------------------


카이제곱 검정 결과 p-value < 0.05(유의수준)이므로

귀무가설을 기각한다.

즉, 기능 경험 여부에 따라 D30 리텐션에 유의미한 차이가 있다.

### 2. 가설설정
귀무가설(H0): 수면기록 이후 다른 기능으로 확장한 여부에 따라
              D30 리텐션에 차이가 없다

대립가설(H1): 수면기록 이후 다른 기능으로 확장한 여부에 따라
              D30 리텐션에 차이가 있다

유의수준: α = 0.05
검정방법: 카이제곱 검정
이유: 확장 여부(범주형) × D30 복귀 여부(범주형)

확장형:
수면기록 O + 운동기록/식단기록/마음챙김/챌린지 중 하나라도 O
→ 수면기록을 시작으로 다른 기능까지 쓴 유저

미확장형:
수면기록 O + 다른 기능 X
→ 수면기록만 쓰고 다른 기능은 안 쓴 유저

In [39]:
# 수면기록 경험 유저
sleep_users = set(event_clean[event_clean['event_type'] == '수면기록']['user_id'])

# 다른 기능도 쓴 유저 (확장형)
other_features = ['운동기록', '식단기록', '마음챙김', '챌린지참여']
expanded_users = set(
    event_clean[
        (event_clean['user_id'].isin(sleep_users)) &
        (event_clean['event_type'].isin(other_features))
    ]['user_id']
)

# 수면기록만 한 유저 (미확장형)
not_expanded_users = sleep_users - expanded_users

print(f"수면기록 경험 유저: {len(sleep_users):,}명")
print(f"확장형 유저: {len(expanded_users):,}명")
print(f"미확장형 유저: {len(not_expanded_users):,}명")

수면기록 경험 유저: 11,432명
확장형 유저: 11,249명
미확장형 유저: 183명


미확장형이 183명으로 샘플이 너무 적어서 통계검정 결과가 나와도 신뢰가....

In [40]:
t1, d1, r1 = get_d30_retention(expanded_users, app_launch, user)
t2, d2, r2 = get_d30_retention(not_expanded_users, app_launch, user)

table = [[d1, t1-d1], [d2, t2-d2]]
chi2, p, dof, expected = chi2_contingency(table)

df = pd.DataFrame({
    '구분': ['확장형', '미확장형'],
    '전체유저수': [t1, t2],
    'D30 유저수': [d1, d2],
    'D30 리텐션(%)': [r1, r2],
})

stat_df = pd.DataFrame({
    '카이제곱 통계량': [round(chi2, 4)],
    '자유도': [dof],
    'p-value': [f'{p:.2e}'],
    '결론': ['유의미 O' if p < 0.05 else '유의미 X']
})

display(df)
display(stat_df)

,구분,전체유저수,D30 유저수,D30 리텐션(%)
0,확장형,11249,2943,26.2
1,미확장형,183,0,0.0


,카이제곱 통계량,자유도,p-value,결론
0,63.1139,1,1.95e-15,유의미 O


수면기록 이후 다른 기능으로 확장한 유저의 D30이 26.2%,

미확장형 유저는 0.0%로 통계적으로 유의미한 차이가 있다.

한계점

미확장형 유저가 183명으로 샘플이 적어

결과 해석 시 주의 필요

### 3. 가설설정
- 귀무가설: 가입경로별 D30 리텐션에 차이가 없다
- 대립가설: 가입경로별 D30 리텐션에 차이가 있다
- 유의수준: α = 0.05
- 검정방법: 카이제곱 검정 (가입경로(범주형) × D30 복귀 여부(범주형))

In [41]:
organic_users = set(user[user['signup_channel'] == '오가닉']['user_id'])
ad_users = set(user[user['signup_channel'] == '퍼포먼스광고']['user_id'])

t1, d1, r1 = get_d30_retention(organic_users, app_launch, user)
t2, d2, r2 = get_d30_retention(ad_users, app_launch, user)

table = [[d1, t1-d1], [d2, t2-d2]]
chi2, p, dof, expected = chi2_contingency(table)

df = pd.DataFrame({
    '구분': ['오가닉', '퍼포먼스광고'],
    '전체유저수': [t1, t2],
    'D30 유저수': [d1, d2],
    'D30 리텐션(%)': [r1, r2],
})

stat_df = pd.DataFrame({
    '카이제곱 통계량': [round(chi2, 4)],
    '자유도': [dof],
    'p-value': [f'{p:.2e}'],
    '결론': ['유의미 O' if p < 0.05 else '유의미 X']
})

display(df)
display(stat_df)

,구분,전체유저수,D30 유저수,D30 리텐션(%)
0,오가닉,5511,1306,23.7
1,퍼포먼스광고,6852,1604,23.4


,카이제곱 통계량,자유도,p-value,결론
0,0.126,1,7.23e-01,유의미 X


p-value = 0.723 > 0.05 → 귀무가설을 기각하지 못한다.

결론:
오가닉과 퍼포먼스광고 유저 간 D30 리텐션 차이(23.7% vs 23.4%)는

통계적으로 유의미하지 않다.

즉, 가입경로는 D30 리텐션에 영향을 미치지 않는다.

광고로 데려온 유저가 오가닉 유저보다 리텐션이 낮을 것이다"

→ 이 가설이 틀렸음을 데이터로 증명

### 4. 가설설정
- 귀무가설: 기기별로 D30 리텐션에 차이가 없다
- 대립가설: 기기별로 D30 리텐션에 차이가 있다
- 유의수준: α = 0.05
- 검정방법: 카이제곱 검정 (기기(범주형) × D30 복귀 여부(범주형))

In [42]:
ios_users = set(user[user['device'] == 'iOS']['user_id'])
android_users = set(user[user['device'] == 'Android']['user_id'])

t1, d1, r1 = get_d30_retention(ios_users, app_launch, user)
t2, d2, r2 = get_d30_retention(android_users, app_launch, user)

table = [[d1, t1-d1], [d2, t2-d2]]
chi2, p, dof, expected = chi2_contingency(table)

df = pd.DataFrame({
    '구분': ['iOS', 'Android'],
    '전체유저수': [t1, t2],
    'D30 유저수': [d1, d2],
    'D30 리텐션(%)': [r1, r2],
})

stat_df = pd.DataFrame({
    '카이제곱 통계량': [round(chi2, 4)],
    '자유도': [dof],
    'p-value': [f'{p:.2e}'],
    '결론': ['유의미 O' if p < 0.05 else '유의미 X']
})

display(df)
display(stat_df)

,구분,전체유저수,D30 유저수,D30 리텐션(%)
0,iOS,7175,1678,23.4
1,Android,5204,1234,23.7


,카이제곱 통계량,자유도,p-value,결론
0,0.1603,1,6.89e-01,유의미 X


p-value = 0.689 > 0.05 → 귀무가설 기각하지 못한다.

결론:
iOS와 Android 유저 간 D30 리텐션 차이(23.4% vs 23.7%)는

통계적으로 유의미하지 않다.

즉, 기기 종류는 D30 리텐션에 영향을 미치지 않는다.

### 지금까지 통계검정한걸 바탕으로 나온 인사이트
가입경로(오가닉 vs 광고)와 기기(iOS vs Android) 모두

D30 리텐션에 유의미한 영향을 미치지 않는다.

즉, 리텐션 하락 원인은 유입 채널이나 플랫폼이 아니라

앱 내부 경험(기능 사용, 챌린지 참여)에 있다.

### 5. 가설설정
- 귀무가설: 월별 코호트에 따라 D30 리텐션에 차이가 없다
- 대립가설: 월별 코호트에 따라 D30 리텐션에 차이가 있다
- 유의수준: α = 0.05
- 검정방법: 카이제곱 검정 (가입월(5그룹) × D30 복귀 여부(범주형))

In [43]:
user['signup_month'] = pd.to_datetime(user['signup_date']).dt.to_period('M').astype(str)

result = []
contingency = []

for month in sorted(user['signup_month'].unique()):
    month_users = set(user[user['signup_month'] == month]['user_id'])
    t, d, r = get_d30_retention(month_users, app_launch, user)
    result.append({'코호트': month, '전체유저수': t, 'D30유저수': d, 'D30리텐션(%)': r})
    contingency.append([d, t - d])

chi2, p, dof, expected = chi2_contingency(contingency)

df = pd.DataFrame(result)
stat_df = pd.DataFrame({
    '카이제곱 통계량': [round(chi2, 4)],
    '자유도': [dof],
    'p-value': [f'{p:.2e}'],
    '결론': ['유의미 O' if p < 0.05 else '유의미 X']
})

display(df)
display(stat_df)

,코호트,전체유저수,D30유저수,D30리텐션(%)
0,2025-01,2124,480,22.6
1,2025-02,4384,1178,26.9
2,2025-03,2122,488,23.0
3,2025-04,2082,432,20.7
4,2025-05,1788,365,20.4


,카이제곱 통계량,자유도,p-value,결론
0,47.1209,4,1.44e-09,유의미 O


p-value = 1.44e-09 < 0.05 → 귀무가설 기각이 가능하다.

결론:
월별 코호트에 따라 D30 리텐션에 통계적으로 유의미한 차이가 있다.

2월 코호트가 26.9%로 가장 높고, 4~5월 코호트는 20%대로 하락 추세를 보인다.

### 한계점
3월 코호트는 로그 수집 장애(3/10~14) 영향으로

실제보다 낮게 측정되었을 가능성 있음

### 6. 가설설정
- 귀무가설: 알림 동의 여부에 따라 D30 리텐션에 차이가 없다
- 대립가설: 알림 동의 여부에 따라 D30 리텐션에 차이가 있다
- 유의수준: α = 0.05
- 검정방법: 카이제곱 검정 (알림동의여부(범주형) × D30 복귀 여부(범주형))

In [44]:
print(user['notification_agreed'].value_counts())
print(user['notification_agreed'].dtype)

notification_agreed
True       7984
False      4400
Unknown     116
Name: count, dtype: int64
str


Unknown 제외 하고 확인 했음.

In [45]:
agreed_users = set(user[user['notification_agreed'] == 'True']['user_id'])
not_agreed_users = set(user[user['notification_agreed'] == 'False']['user_id'])

print(f"알림 동의: {len(agreed_users):,}명")
print(f"알림 미동의: {len(not_agreed_users):,}명")

t1, d1, r1 = get_d30_retention(agreed_users, app_launch, user)
t2, d2, r2 = get_d30_retention(not_agreed_users, app_launch, user)

알림 동의: 7,984명
알림 미동의: 4,400명


In [46]:
agreed_users = set(user[user['notification_agreed'] == 'True']['user_id'])
not_agreed_users = set(user[user['notification_agreed'] == 'False']['user_id'])

t1, d1, r1 = get_d30_retention(agreed_users, app_launch, user)
t2, d2, r2 = get_d30_retention(not_agreed_users, app_launch, user)

table = [[d1, t1-d1], [d2, t2-d2]]
chi2, p, dof, expected = chi2_contingency(table)

df = pd.DataFrame({
    '구분': ['알림 동의', '알림 미동의'],
    '전체유저수': [t1, t2],
    'D30 유저수': [d1, d2],
    'D30 리텐션(%)': [r1, r2],
})

stat_df = pd.DataFrame({
    '카이제곱 통계량': [round(chi2, 4)],
    '자유도': [dof],
    'p-value': [f'{p:.2e}'],
    '결론': ['유의미 O' if p < 0.05 else '귀무가설 기각 못함 X']
})

display(df)
display(stat_df)

,구분,전체유저수,D30 유저수,D30 리텐션(%)
0,알림 동의,7984,1836,23.0
1,알림 미동의,4400,1080,24.5


,카이제곱 통계량,자유도,p-value,결론
0,3.6975,1,5.45e-02,귀무가설 기각 못함 X


p-value = 0.0545 > 0.05 → 귀무가설 기각하지 못함(거의 가까운 결과값...)

알림 미동의 유저가 오히려 동의 유저보다 d30이 1.5%정도 약간 높네... 통계적으로 유의미하지 않다로 봐도 되는건가??

한계점:
p-value가 0.055로 유의수준 0.05에 매우 근접해

유의수준을 0.1로 설정했다면 유의미했을 수 있음.

# 7. 가설설정
- 귀무가설: 온보딩 완료 후 첫 활동 여부에 따라 D30 리텐션에 차이가 없다
- 대립가설: 온보딩 완료 후 첫 활동 여부에 따라 D30 리텐션에 차이가 있다
- 유의수준: α = 0.05
- 검정방법: 카이제곱 검정 (첫활동여부(범주형) × D30 복귀 여부(범주형))

In [47]:
activity_types = ['수면기록', '운동기록', '식단기록', '마음챙김', '챌린지참여']

# 온보딩 완료 유저
completed_ids = set(user[user['is_onboarding_completed'] == True]['user_id'])

# 온보딩 완료 유저의 첫 활동 경험 여부
first_activity_ids = set(
    event_clean[
        (event_clean['user_id'].isin(completed_ids)) &
        (event_clean['event_type'].isin(activity_types))
    ]['user_id']
)

# 그룹 분류
completed_with_activity = first_activity_ids
completed_no_activity = completed_ids - first_activity_ids
not_completed = set(user['user_id']) - completed_ids

t1, d1, r1 = get_d30_retention(completed_with_activity, app_launch, user)
t2, d2, r2 = get_d30_retention(completed_no_activity, app_launch, user)
t3, d3, r3 = get_d30_retention(not_completed, app_launch, user)

table = [[d1, t1-d1], [d2, t2-d2], [d3, t3-d3]]
chi2, p, dof, expected = chi2_contingency(table)

df = pd.DataFrame({
    '구분': ['온보딩완료+첫활동O', '온보딩완료+첫활동X', '온보딩미완료'],
    '전체유저수': [t1, t2, t3],
    'D30 유저수': [d1, d2, d3],
    'D30 리텐션(%)': [r1, r2, r3],
})

stat_df = pd.DataFrame({
    '카이제곱 통계량': [round(chi2, 4)],
    '자유도': [dof],
    'p-value': [f'{p:.2e}'],
    '결론': ['유의미 O' if p < 0.05 else '귀무가설 기각 못함 X']
})

display(df)
display(stat_df)

,구분,전체유저수,D30 유저수,D30 리텐션(%)
0,온보딩완료+첫활동O,5658,1621,28.6
1,온보딩완료+첫활동X,61,0,0.0
2,온보딩미완료,6781,1322,19.5


,카이제곱 통계량,자유도,p-value,결론
0,162.4605,2,5.27e-36,유의미 O


p-value = 0.00 < 0.05 → 귀무가설 기각 가능

카이제곱 통계량 = 2286.71 → 차이가 매우 크다는 뜻

결론:
온보딩 완료 후 첫 활동 여부에 따라 D30 리텐션에 통계적으로 유의미한 차이가 있다.

온보딩 완료 후 첫 활동을 한 유저(28.6%)가 미완료 유저(19.5%)보다 높으며,

온보딩 완료 후 첫 활동을 하지 않은 유저는 D30이 0.0%다.

한계점: 온보딩완료+첫활동X 61명으로 샘플이 적어 해석 주의 필요

### 8. 가설설정
- 귀무가설: 온보딩+챌린지 조합에 따라 D30 리텐션에 차이가 없다
- 대립가설: 온보딩+챌린지 조합에 따라 D30 리텐션에 차이가 있다
- 유의수준: α = 0.05
- 검정방법: 카이제곱 검정 (4그룹 × D30 복귀 여부)

In [48]:
challenge_ids = set(event_clean[event_clean['event_type'] == '챌린지참여']['user_id'])
completed_ids = set(user[user['is_onboarding_completed'] == True]['user_id'])
all_user_ids = set(user['user_id'])

group1 = completed_ids & challenge_ids          # 온보딩완료+챌린지O
group2 = completed_ids - challenge_ids          # 온보딩완료+챌린지X
group3 = (all_user_ids - completed_ids) & challenge_ids  # 온보딩미완료+챌린지O
group4 = (all_user_ids - completed_ids) - challenge_ids  # 온보딩미완료+챌린지X

t1, d1, r1 = get_d30_retention(group1, app_launch, user)
t2, d2, r2 = get_d30_retention(group2, app_launch, user)
t3, d3, r3 = get_d30_retention(group3, app_launch, user)
t4, d4, r4 = get_d30_retention(group4, app_launch, user)

table = [[d1, t1-d1], [d2, t2-d2], [d3, t3-d3], [d4, t4-d4]]
chi2, p, dof, expected = chi2_contingency(table)

df = pd.DataFrame({
    '구분': ['온보딩완료+챌린지O', '온보딩완료+챌린지X', '온보딩미완료+챌린지O', '온보딩미완료+챌린지X'],
    '전체유저수': [t1, t2, t3, t4],
    'D30 유저수': [d1, d2, d3, d4],
    'D30 리텐션(%)': [r1, r2, r3, r4],
})

stat_df = pd.DataFrame({
    '카이제곱 통계량': [round(chi2, 4)],
    '자유도': [dof],
    'p-value': [f'{p:.2e}'],
    '결론': ['유의미 O' if p < 0.05 else '귀무가설 기각 못함 X']
})

display(df)
display(stat_df)

,구분,전체유저수,D30 유저수,D30 리텐션(%)
0,온보딩완료+챌린지O,4928,1615,32.8
1,온보딩완료+챌린지X,791,6,0.8
2,온보딩미완료+챌린지O,4393,1315,29.9
3,온보딩미완료+챌린지X,2388,7,0.3


,카이제곱 통계량,자유도,p-value,결론
0,1278.0791,3,8.40e-277,유의미 O


p-value = 8.40e-277 < 0.05 → 귀무가설 기각가능

결론: 챌린지 참여 여부가 온보딩 완료 여부보다 D30에 더 결정적이다.

온보딩 미완료여도 챌린지 참여 시 D30 29.9%,

온보딩 완료해도 챌린지 미참여 시 D30 0.8%

온보딩+챌린지 조합에 따라 D30 리텐션에 통계적으로 유의미한 차이가 있다.

챌린지 참여가 D30 리텐션의 핵심 드라이버임이 통계적으로 검증완료.